# AIaaS — Peak-Ceiling LLM Benchmark (TensorRT-LLM)

The *peak hardware* tier: the best-case LLM serving throughput an NVIDIA GPU can
hit, using **TensorRT-LLM**'s own benchmark CLI **`trtllm-bench`** (PyTorch
backend, CUDA graphs on by default — no manual engine build).

Pairs directly with `vllm_serving_benchmark.ipynb`: same workload (LLM decode over
a token-length distribution), so the gap between the two is the **vLLM → TensorRT-LLM
headroom** on your hardware. With `--streaming` it reports the same family of
metrics — **TTFT / TPOT / output throughput**.

### How it references the fork
Dataset generation uses **`benchmarks/cpp/prepare_dataset.py` from your
`kurtvalcorza/TensorRT-LLM` fork** (shallow-cloned), so the workload is produced
exactly the way TensorRT-LLM's own performance docs do it.

### Requirements
- **A100 / Hopper-class GPU.** Recent TensorRT-LLM targets Ampere+ — **Colab T4
  (Turing) is not supported**; use the vLLM notebook there instead.
- TensorRT-LLM installs a large CUDA stack; expect a slow first install and a
  possible runtime restart.

> Per the strategy doc, the real runs live in a session scoped to the
> `TensorRT-LLM` fork. This notebook is portable so it can be dropped in there.
> `trtllm-bench` flag names drift between releases — if a flag errors, check
> `trtllm-bench throughput -h` and adjust the small config cell.

## 1. Install TensorRT-LLM

In [ ]:
import subprocess, sys

# TensorRT-LLM wheels are published on NVIDIA's index. Large download.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "tensorrt-llm", "pandas",
                "--extra-index-url", "https://pypi.nvidia.com"], check=False)

import shutil
print("trtllm-bench on PATH:", shutil.which("trtllm-bench"))
try:
    import tensorrt_llm
    print("tensorrt_llm", tensorrt_llm.__version__)
except Exception as e:
    print("tensorrt_llm not importable yet:", e)
    print("If a CUDA lib was upgraded, RESTART the runtime and re-run from this cell.")


## 2. Environment & hardware capture

In [ ]:
import os, json, platform, subprocess, datetime
import torch

def smi(field):
    try:
        out = subprocess.run(["nvidia-smi", f"--query-gpu={field}",
                              "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10)
        return [x.strip() for x in out.stdout.strip().splitlines() if x.strip()]
    except Exception:
        return []

def detect_platform():
    if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ: return "colab"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ: return "kaggle"
    if os.path.exists("/opt/.sagemakerinternal"): return "sagemaker-studio-lab"
    return "local/on-prem"

CUDA = torch.cuda.is_available()
assert CUDA, "No CUDA GPU detected. This benchmark requires a GPU runtime."

_cc = torch.cuda.get_device_capability(0)
if _cc[0] < 8:
    print(f"WARNING: compute capability {_cc[0]}.{_cc[1]} (<8.0). Recent "
          "TensorRT-LLM targets Ampere+; this GPU may be unsupported. "
          "Use the vLLM notebook here instead.")

try:
    import tensorrt_llm; trtllm_ver = tensorrt_llm.__version__
except Exception:
    trtllm_ver = "?"

ENV = {
    "platform": detect_platform(),
    "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_count": torch.cuda.device_count(),
    "vram_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
    "compute_capability": f"{_cc[0]}.{_cc[1]}",
    "cuda": torch.version.cuda,
    "driver": (smi("driver_version") or ["?"])[0],
    "torch": torch.__version__,
    "tensorrt_llm": trtllm_ver,
    "python": platform.python_version(),
}
print(json.dumps(ENV, indent=2))


## 3. Configuration
Ungated Qwen2.5, VRAM-tiered. The input/output lengths define the synthetic
token-length distribution; the concurrency sweep traces the throughput curve.

In [ ]:
VRAM = ENV["vram_total_gb"]
TIER = "large" if VRAM >= 24 else "small"
MODEL = "Qwen/Qwen2.5-7B-Instruct" if TIER == "large" else "Qwen/Qwen2.5-1.5B-Instruct"

BACKEND = "pytorch"        # PyTorch flow: no separate engine build
TP_SIZE = 1               # tensor-parallel GPUs; recorded for the cost model
STREAMING = True          # needed for TTFT / TPOT (per-token) metrics

INPUT_LEN = 1024          # mean prompt tokens (stdev 0 -> fixed length)
OUTPUT_LEN = 1024         # mean generated tokens
NUM_REQUESTS = 1000       # dataset size
CONCURRENCIES = [1, 16, 256]   # in-flight requests per run -> latency/throughput curve

REPO_URL = "https://github.com/kurtvalcorza/TensorRT-LLM"   # your fork (reference)
REPO_DIR = "TensorRT-LLM-ref"
OUT_DIR = "trtllm_bench_results"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"VRAM={VRAM} GB -> TIER={TIER}, MODEL={MODEL}, BACKEND={BACKEND}, "
      f"TP_SIZE={TP_SIZE}, ISL/OSL={INPUT_LEN}/{OUTPUT_LEN}")


## 4. Dataset — via the fork's `prepare_dataset.py`
Shallow-clone the fork only to reuse its official dataset generator, then build a
token-normalized synthetic dataset (fixed input/output lengths).

In [ ]:
import subprocess, os

if not os.path.isdir(REPO_DIR):
    print("Cloning fork (shallow) for prepare_dataset.py ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

PREP = os.path.join(REPO_DIR, "benchmarks", "cpp", "prepare_dataset.py")
assert os.path.exists(PREP), f"prepare_dataset.py not found at {PREP}"

DATASET_FILE = os.path.join(OUT_DIR, f"synthetic_{INPUT_LEN}_{OUTPUT_LEN}.txt")
prep_cmd = [
    sys.executable, PREP, "--stdout", "--tokenizer", MODEL,
    "token-norm-dist",
    "--input-mean", str(INPUT_LEN), "--output-mean", str(OUTPUT_LEN),
    "--input-stdev", "0", "--output-stdev", "0",
    "--num-requests", str(NUM_REQUESTS),
]
print("Generating dataset:", " ".join(prep_cmd))
with open(DATASET_FILE, "w") as f:
    p = subprocess.run(prep_cmd, stdout=f, stderr=subprocess.PIPE, text=True)
if p.returncode != 0:
    print("prepare_dataset stderr:\n", p.stderr[-2000:])
    raise RuntimeError("dataset generation failed")
print("Wrote", DATASET_FILE, "-", os.path.getsize(DATASET_FILE), "bytes")


## 5. Run `trtllm-bench throughput` over the concurrency sweep
Each concurrency point runs the PyTorch backend at that in-flight load and writes
a structured JSON report (with a stdout fallback if `--report_json` is absent).

In [ ]:
import subprocess, os, json, re

def parse_stdout(text):
    """Fallback: scrape 'Label: number' pairs from the perf overview block."""
    metrics = {}
    for line in text.splitlines():
        m = re.match(r"\s*([A-Za-z][A-Za-z0-9 /()\-\.]+?):\s+([-\d.]+)\s*$", line)
        if m:
            try:
                metrics[m.group(1).strip()] = float(m.group(2))
            except ValueError:
                pass
    return metrics

def run_point(concurrency):
    report_json = os.path.join(OUT_DIR, f"report_c{concurrency}.json")
    if os.path.exists(report_json):
        os.remove(report_json)
    cmd = [
        "trtllm-bench", "--model", MODEL,
        "throughput", "--dataset", DATASET_FILE,
        "--backend", BACKEND,
        "--num_requests", str(NUM_REQUESTS),
        "--concurrency", str(concurrency),
        "--report_json", report_json,
    ]
    if STREAMING:
        cmd.append("--streaming")
    if TP_SIZE > 1:
        cmd += ["--tp", str(TP_SIZE)]   # flag name may be --tp_size on some releases
    print("\n=== concurrency =", concurrency, "===")
    print(" ".join(cmd))
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(p.stdout[-2000:])
    if p.returncode != 0:
        print("STDERR:", p.stderr[-2000:])
        return {"error": f"trtllm-bench exited {p.returncode}", "stderr_tail": p.stderr[-800:]}
    if os.path.exists(report_json):
        with open(report_json) as f:
            return {"report_json": json.load(f)}
    return {"parsed_stdout": parse_stdout(p.stdout)}

SWEEP = {}
for c in CONCURRENCIES:
    SWEEP[str(c)] = run_point(c)


## 6. Results — normalize, save, summarize
TensorRT-LLM's report schema varies by release, so headline metrics are plucked
defensively from either the JSON report or the stdout scrape; the full payload is
always saved.

In [ ]:
import pandas as pd

def pluck(d, *keys):
    """Return the first key present (case-insensitive) in dict d."""
    if not isinstance(d, dict):
        return None
    low = {k.lower(): v for k, v in d.items()}
    for k in keys:
        if k.lower() in low:
            return low[k.lower()]
    return None

def summarize(point):
    if "error" in point:
        return {"status": point["error"][:40]}
    rep = point.get("report_json") or {}
    perf = rep.get("performance") or rep.get("performance_overview") or rep
    flat = point.get("parsed_stdout") or {}
    def grab(*names):
        return pluck(perf, *names) or pluck(flat, *names)
    return {
        "out tok/s": grab("total_output_throughput_tps", "Total Output Throughput (tokens/sec)",
                          "output_throughput", "Token Throughput (tokens/sec)"),
        "req/s": grab("request_throughput", "Request Throughput (req/sec)"),
        "TTFT ms (p50)": grab("ttft_p50_ms", "average_time_to_first_token_ms",
                              "Average time-to-first-token (ms)"),
        "TPOT ms": grab("tpot_ms", "average_time_per_output_token_ms",
                       "Per User Output Throughput (tokens/sec/user)"),
    }

SUMMARY = {c: summarize(p) for c, p in SWEEP.items()}

NORM = {
    "schema": "trtllm-bench/1.0",
    "engine": "tensorrt-llm",
    "env": ENV, "tier": TIER, "model": MODEL,
    "backend": BACKEND, "tensor_parallel_size": TP_SIZE,
    "input_len": INPUT_LEN, "output_len": OUTPUT_LEN,
    "num_requests": NUM_REQUESTS, "concurrencies": CONCURRENCIES,
    "summary": SUMMARY, "sweep": SWEEP,
}
tag = (ENV["platform"] + "_" + ENV["gpu_name"]).replace(" ", "-").replace("/", "-")
out = os.path.join(OUT_DIR, f"trtllm_bench_{tag}.json")
with open(out, "w") as f:
    json.dump(NORM, f, indent=2)

print("Saved:", out)
print("\n==== PEAK-CEILING SUMMARY (TensorRT-LLM) ====")
print(f"{ENV['gpu_name']} ({ENV['vram_total_gb']} GB) x{TP_SIZE} | {MODEL} | "
      f"ISL/OSL={INPUT_LEN}/{OUTPUT_LEN}")
df = pd.DataFrame(SUMMARY).T
df.index.name = "concurrency"
try:
    from IPython.display import display
    display(df)
except Exception:
    print(df.to_string())


## 7. Reading the result (and what to compare)

- The **highest-concurrency** row is the peak output throughput ceiling; the
  **concurrency=1** row is the best-case single-stream latency (TTFT / TPOT).
- **Compare to the vLLM notebook** at a matching input/output length: TensorRT-LLM
  is usually faster — the delta is the headroom from the most optimized engine on
  this GPU. (Match `INPUT_LEN`/`OUTPUT_LEN` here to the vLLM run's distribution
  for a fair comparison; vLLM uses ShareGPT by default, so use its `random`
  dataset mode with the same lengths for an apples-to-apples line-up.)
- A row showing `status` instead of numbers means that concurrency point failed —
  the others are still valid.

See `BENCHMARKING_STRATEGY.md`. The actual runs belong in a session scoped to the
`kurtvalcorza/TensorRT-LLM` fork.